<a href="https://colab.research.google.com/github/Nithin-Adithya/cuda-sgemm/blob/main/cuda_sgemm_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CUDA SGEMM: staged optimization vs cuBLAS

**First:** Runtime > Change runtime type > **T4 GPU** (or any GPU), then Runtime > Run all.

Stages: naive, coalesced, shared-memory tiled, register-blocked, vectorized loads.

In [ ]:
!rm -rf /content/v && git clone -q https://github.com/Nithin-Adithya/cuda-sgemm /content/v && cd /content/v && git log --oneline | head -1 && mkdir -p build && nvcc -O3 -std=c++17 -lineinfo src/main.cu -o build/sgemm -lcublas && ./build/sgemm --out /content/v/results

## 1. Check the GPU and toolchain

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!nvcc --version | tail -2

## 2. Write the sources

In [ ]:
import os
os.makedirs('/content/cuda-sgemm', exist_ok=True)
%cd /content/cuda-sgemm
!mkdir -p src scripts build results

In [ ]:
%%writefile src/kernels.cuh
// SGEMM kernels, one per optimization stage. C (MxN) = A (MxK) * B (KxN),
// all matrices row-major, single precision.
//
// This header is compiled by nvcc for the GPU build and by a host compiler for
// the CPU emulation test (test/cuda_emu.h supplies the CUDA builtins there), so
// it must only use: __global__, __shared__, __syncthreads, threadIdx, blockIdx,
// float4 / make_float4.
#pragma once

// ---------------------------------------------------------------------------
// Stage 0: naive. threadIdx.x indexes the *row*, so the 32 threads of a warp
// touch 32 different rows: A reads are strided by K and C writes by N. Every
// warp-level access becomes 32 separate memory transactions.
// Launch: block (B, B), grid (ceil(M/B), ceil(N/B)).
// ---------------------------------------------------------------------------
__global__ void sgemm_naive(int M, int N, int K, const float *A, const float *B, float *C)
{
    const int row = blockIdx.x * blockDim.x + threadIdx.x;
    const int col = blockIdx.y * blockDim.y + threadIdx.y;
    if (row >= M || col >= N) return;

    float acc = 0.0f;
    for (int k = 0; k < K; ++k)
        acc += A[row * K + k] * B[k * N + col];
    C[row * N + col] = acc;
}

// ---------------------------------------------------------------------------
// Stage 1: global-memory coalescing. Same arithmetic, but threadIdx.x now
// indexes the *column*. Within a warp, A[row*K + k] is a single address
// (broadcast), B[k*N + col] is 32 consecutive floats (one 128-byte
// transaction), and the C write is consecutive too.
// Launch: block (B, B), grid (ceil(N/B), ceil(M/B)).
// ---------------------------------------------------------------------------
__global__ void sgemm_coalesced(int M, int N, int K, const float *A, const float *B, float *C)
{
    const int col = blockIdx.x * blockDim.x + threadIdx.x;
    const int row = blockIdx.y * blockDim.y + threadIdx.y;
    if (row >= M || col >= N) return;

    float acc = 0.0f;
    for (int k = 0; k < K; ++k)
        acc += A[row * K + k] * B[k * N + col];
    C[row * N + col] = acc;
}

// ---------------------------------------------------------------------------
// Stage 2: shared-memory tiling. Each block computes a TILE x TILE patch of C.
// Per K-step it stages a TILE x TILE tile of A and of B in shared memory, so
// each global element is fetched once per block instead of once per thread.
// Requires M, N, K to be multiples of TILE.
// Launch: block (TILE, TILE), grid (N/TILE, M/TILE).
// ---------------------------------------------------------------------------
template <int TILE>
__global__ void sgemm_tiled(int M, int N, int K, const float *A, const float *B, float *C)
{
    __shared__ float As[TILE * TILE];
    __shared__ float Bs[TILE * TILE];

    const int tx = threadIdx.x, ty = threadIdx.y;
    const int row = blockIdx.y * TILE + ty;
    const int col = blockIdx.x * TILE + tx;

    float acc = 0.0f;
    for (int k0 = 0; k0 < K; k0 += TILE) {
        As[ty * TILE + tx] = A[row * K + k0 + tx];
        Bs[ty * TILE + tx] = B[(k0 + ty) * N + col];
        __syncthreads();

        for (int k = 0; k < TILE; ++k)
            acc += As[ty * TILE + k] * Bs[k * TILE + tx];
        __syncthreads();
    }
    C[row * N + col] = acc;
}

// ---------------------------------------------------------------------------
// Stage 3: register blocking. A block still owns a BM x BN patch of C, but
// each thread now computes a TM x TN sub-patch and keeps it in registers.
// Per k it reads TM values of A and TN values of B from shared memory and
// does TM*TN FMAs, so shared-memory loads per FMA drop from 2 to
// (TM+TN)/(TM*TN). That moves the bottleneck from memory to arithmetic.
// Requires M % BM == 0, N % BN == 0, K % BK == 0.
// Launch: block ((BM/TM)*(BN/TN)), grid (N/BN, M/BM).
// ---------------------------------------------------------------------------
template <int BM, int BN, int BK, int TM, int TN>
__global__ void sgemm_regblock(int M, int N, int K, const float *A, const float *B, float *C)
{
    __shared__ float As[BM * BK];
    __shared__ float Bs[BK * BN];

    const int nthreads = (BM / TM) * (BN / TN);
    const int tid = threadIdx.x;
    const int tcol = tid % (BN / TN);
    const int trow = tid / (BN / TN);

    // Move to this block's patch.
    A += blockIdx.y * BM * K;
    B += blockIdx.x * BN;
    C += blockIdx.y * BM * N + blockIdx.x * BN;

    float acc[TM * TN] = {0.0f};
    float ra[TM], rb[TN];

    for (int k0 = 0; k0 < K; k0 += BK) {
        for (int i = tid; i < BM * BK; i += nthreads)
            As[i] = A[(i / BK) * K + k0 + (i % BK)];
        for (int i = tid; i < BK * BN; i += nthreads)
            Bs[i] = B[(k0 + i / BN) * N + (i % BN)];
        __syncthreads();

        for (int k = 0; k < BK; ++k) {
            for (int i = 0; i < TM; ++i) ra[i] = As[(trow * TM + i) * BK + k];
            for (int j = 0; j < TN; ++j) rb[j] = Bs[k * BN + tcol * TN + j];
            for (int i = 0; i < TM; ++i)
                for (int j = 0; j < TN; ++j)
                    acc[i * TN + j] += ra[i] * rb[j];
        }
        __syncthreads();
    }

    for (int i = 0; i < TM; ++i)
        for (int j = 0; j < TN; ++j)
            C[(trow * TM + i) * N + tcol * TN + j] = acc[i * TN + j];
}

// ---------------------------------------------------------------------------
// Stage 4: vectorized loads. Same blocking as stage 3, plus:
//   * global -> shared copies go through float4 (one 128-bit LDG per 4 floats),
//   * the A tile is stored transposed (As[k][row]) so the per-k register fill
//     reads contiguous shared memory, which the compiler can also vectorize,
//   * C is written back as float4.
// Requires the stage 3 divisibility, plus BK % 4 == 0, TN % 4 == 0, and
// K % 4 == 0, N % 4 == 0 so every float4 access is 16-byte aligned.
// Launch: block ((BM/TM)*(BN/TN)), grid (N/BN, M/BM).
// ---------------------------------------------------------------------------
template <int BM, int BN, int BK, int TM, int TN>
__global__ void sgemm_vectorized(int M, int N, int K, const float *A, const float *B, float *C)
{
    __shared__ float As[BK * BM]; // transposed: As[k * BM + row]
    __shared__ float Bs[BK * BN];

    const int nthreads = (BM / TM) * (BN / TN);
    const int tid = threadIdx.x;
    const int tcol = tid % (BN / TN);
    const int trow = tid / (BN / TN);

    A += blockIdx.y * BM * K;
    B += blockIdx.x * BN;
    C += blockIdx.y * BM * N + blockIdx.x * BN;

    float acc[TM * TN] = {0.0f};
    float ra[TM], rb[TN];

    for (int k0 = 0; k0 < K; k0 += BK) {
        for (int i = tid; i < BM * BK / 4; i += nthreads) {
            const int r = i / (BK / 4);
            const int c = (i % (BK / 4)) * 4;
            const float4 v = reinterpret_cast<const float4 *>(&A[r * K + k0 + c])[0];
            As[(c + 0) * BM + r] = v.x;
            As[(c + 1) * BM + r] = v.y;
            As[(c + 2) * BM + r] = v.z;
            As[(c + 3) * BM + r] = v.w;
        }
        for (int i = tid; i < BK * BN / 4; i += nthreads) {
            const int r = i / (BN / 4);
            const int c = (i % (BN / 4)) * 4;
            reinterpret_cast<float4 *>(&Bs[r * BN + c])[0] =
                reinterpret_cast<const float4 *>(&B[(k0 + r) * N + c])[0];
        }
        __syncthreads();

        for (int k = 0; k < BK; ++k) {
            for (int i = 0; i < TM; ++i) ra[i] = As[k * BM + trow * TM + i];
            for (int j = 0; j < TN; ++j) rb[j] = Bs[k * BN + tcol * TN + j];
            for (int i = 0; i < TM; ++i)
                for (int j = 0; j < TN; ++j)
                    acc[i * TN + j] += ra[i] * rb[j];
        }
        __syncthreads();
    }

    for (int i = 0; i < TM; ++i)
        for (int j = 0; j < TN; j += 4)
            reinterpret_cast<float4 *>(&C[(trow * TM + i) * N + tcol * TN + j])[0] =
                make_float4(acc[i * TN + j], acc[i * TN + j + 1],
                            acc[i * TN + j + 2], acc[i * TN + j + 3]);
}


In [ ]:
%%writefile src/main.cu
// Benchmark driver: times each SGEMM stage against cuBLAS, checks the result,
// and measures the two roofline ceilings (DRAM bandwidth, peak FMA rate).
//
//   sgemm [--sizes 1024,2048,4096] [--kernel NAME] [--reps 10] [--out results]
//   sgemm --profile --kernel NAME --sizes 2048     (single launch, for ncu)
#include <cuda_runtime.h>
#include <cublas_v2.h>

#include <algorithm>
#include <cmath>
#include <cstdio>
#include <cstdlib>
#include <filesystem>
#include <random>
#include <sstream>
#include <string>
#include <vector>

#include "kernels.cuh"

#define CHECK_CUDA(call)                                                              \
    do {                                                                              \
        cudaError_t err_ = (call);                                                    \
        if (err_ != cudaSuccess) {                                                    \
            std::fprintf(stderr, "CUDA error %s at %s:%d\n", cudaGetErrorString(err_), \
                         __FILE__, __LINE__);                                         \
            std::exit(1);                                                             \
        }                                                                             \
    } while (0)

#define CHECK_CUBLAS(call)                                                         \
    do {                                                                           \
        cublasStatus_t st_ = (call);                                               \
        if (st_ != CUBLAS_STATUS_SUCCESS) {                                        \
            std::fprintf(stderr, "cuBLAS error %d at %s:%d\n", (int)st_, __FILE__, \
                         __LINE__);                                                \
            std::exit(1);                                                          \
        }                                                                          \
    } while (0)

// ---------------------------------------------------------------------------
// Stage table
// ---------------------------------------------------------------------------
struct Problem {
    int M, N, K;
    const float *A, *B;
    float *C;
};

static cublasHandle_t g_cublas = nullptr;

static dim3 grid2d(int x, int y, int b) { return dim3((x + b - 1) / b, (y + b - 1) / b); }

static void run_naive(const Problem &p)
{
    sgemm_naive<<<grid2d(p.M, p.N, 32), dim3(32, 32)>>>(p.M, p.N, p.K, p.A, p.B, p.C);
}
static void run_coalesced(const Problem &p)
{
    sgemm_coalesced<<<grid2d(p.N, p.M, 32), dim3(32, 32)>>>(p.M, p.N, p.K, p.A, p.B, p.C);
}
static void run_tiled(const Problem &p)
{
    sgemm_tiled<32><<<dim3(p.N / 32, p.M / 32), dim3(32, 32)>>>(p.M, p.N, p.K, p.A, p.B, p.C);
}

// Block/thread tile shape shared by the last two stages.
enum { BM = 128, BN = 128, BK = 8, TM = 8, TN = 8 };

static void run_regblock(const Problem &p)
{
    sgemm_regblock<BM, BN, BK, TM, TN><<<dim3(p.N / BN, p.M / BM), dim3((BM / TM) * (BN / TN))>>>(
        p.M, p.N, p.K, p.A, p.B, p.C);
}
static void run_vectorized(const Problem &p)
{
    sgemm_vectorized<BM, BN, BK, TM, TN><<<dim3(p.N / BN, p.M / BM), dim3((BM / TM) * (BN / TN))>>>(
        p.M, p.N, p.K, p.A, p.B, p.C);
}
static void run_cublas(const Problem &p)
{
    // cuBLAS is column-major. Our row-major A, B, C are its A^T, B^T, C^T, and
    // C^T = B^T * A^T, so swap the operands.
    const float one = 1.0f, zero = 0.0f;
    CHECK_CUBLAS(cublasSgemm(g_cublas, CUBLAS_OP_N, CUBLAS_OP_N, p.N, p.M, p.K, &one, p.B, p.N,
                             p.A, p.K, &zero, p.C, p.N));
}

struct Stage {
    const char *name;
    void (*run)(const Problem &);
    int multiple; // M, N, K must be multiples of this
};

static const Stage kStages[] = {
    {"cublas", run_cublas, 1}, // first: it is the reference for the others
    {"naive", run_naive, 1},
    {"coalesced", run_coalesced, 1},
    {"tiled", run_tiled, 32},
    {"regblock", run_regblock, 128},
    {"vectorized", run_vectorized, 128},
};

// ---------------------------------------------------------------------------
// Timing
// ---------------------------------------------------------------------------
struct Timing {
    double best_ms, median_ms;
};

// Times fn() with CUDA events. Stops early once `budget_ms` of GPU time is
// spent (the slow stages take seconds per run at large N), but always does at
// least two reps.
template <typename Fn>
static Timing time_gpu(Fn fn, int reps, double budget_ms = 3000.0)
{
    cudaEvent_t t0, t1;
    CHECK_CUDA(cudaEventCreate(&t0));
    CHECK_CUDA(cudaEventCreate(&t1));
    std::vector<double> ms;
    double total = 0.0;
    for (int r = 0; r < reps && (r < 2 || total < budget_ms); ++r) {
        CHECK_CUDA(cudaEventRecord(t0));
        fn();
        CHECK_CUDA(cudaEventRecord(t1));
        CHECK_CUDA(cudaEventSynchronize(t1));
        float t = 0.0f;
        CHECK_CUDA(cudaEventElapsedTime(&t, t0, t1));
        ms.push_back(t);
        total += t;
    }
    CHECK_CUDA(cudaEventDestroy(t0));
    CHECK_CUDA(cudaEventDestroy(t1));
    std::sort(ms.begin(), ms.end());
    return {ms.front(), ms[ms.size() / 2]};
}

// ---------------------------------------------------------------------------
// Roofline ceilings, measured on this device
// ---------------------------------------------------------------------------
__global__ void triad(float *c, const float *a, const float *b, float s, size_t n)
{
    const size_t i = (size_t)blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) c[i] = a[i] + s * b[i];
}

// Eight independent FMA chains per thread, all in registers: no memory traffic,
// so the rate is the arithmetic ceiling.
__global__ void fma_burn(float *out, float a, float b, int iters)
{
    const int tid = blockIdx.x * blockDim.x + threadIdx.x;
    float x0 = tid * 1e-7f, x1 = x0 + 1, x2 = x0 + 2, x3 = x0 + 3;
    float x4 = x0 + 4, x5 = x0 + 5, x6 = x0 + 6, x7 = x0 + 7;
    for (int i = 0; i < iters; ++i) {
        x0 = x0 * a + b; x1 = x1 * a + b; x2 = x2 * a + b; x3 = x3 * a + b;
        x4 = x4 * a + b; x5 = x5 * a + b; x6 = x6 * a + b; x7 = x7 * a + b;
    }
    out[tid] = x0 + x1 + x2 + x3 + x4 + x5 + x6 + x7;
}

static double measure_bandwidth_gbs()
{
    size_t free_b = 0, total_b = 0;
    CHECK_CUDA(cudaMemGetInfo(&free_b, &total_b));
    size_t n = (size_t)1 << 26; // 256 MB per array: far larger than any cache
    while (3 * n * sizeof(float) > free_b / 2) n >>= 1;

    float *a, *b, *c;
    CHECK_CUDA(cudaMalloc(&a, n * sizeof(float)));
    CHECK_CUDA(cudaMalloc(&b, n * sizeof(float)));
    CHECK_CUDA(cudaMalloc(&c, n * sizeof(float)));
    CHECK_CUDA(cudaMemset(a, 0, n * sizeof(float)));
    CHECK_CUDA(cudaMemset(b, 0, n * sizeof(float)));
    const unsigned blocks = (unsigned)((n + 255) / 256);
    triad<<<blocks, 256>>>(c, a, b, 2.0f, n); // warm-up
    const Timing t = time_gpu([&] { triad<<<blocks, 256>>>(c, a, b, 2.0f, n); }, 10);
    CHECK_CUDA(cudaFree(a));
    CHECK_CUDA(cudaFree(b));
    CHECK_CUDA(cudaFree(c));
    return 3.0 * n * sizeof(float) / (t.best_ms * 1e-3) / 1e9; // 2 reads + 1 write
}

static double measure_peak_gflops(const cudaDeviceProp &prop)
{
    const int blocks = prop.multiProcessorCount * 32, threads = 256, iters = 1 << 16;
    float *out;
    CHECK_CUDA(cudaMalloc(&out, (size_t)blocks * threads * sizeof(float)));
    fma_burn<<<blocks, threads>>>(out, 0.999f, 0.001f, iters); // warm-up
    const Timing t = time_gpu([&] { fma_burn<<<blocks, threads>>>(out, 0.999f, 0.001f, iters); }, 10);
    CHECK_CUDA(cudaFree(out));
    const double flops = 2.0 * 8.0 * iters * (double)blocks * threads; // FMA = 2 flops
    return flops / (t.best_ms * 1e-3) / 1e9;
}

// ---------------------------------------------------------------------------
// Main
// ---------------------------------------------------------------------------
static double rel_error(const std::vector<float> &x, const std::vector<float> &ref)
{
    double num = 0.0, den = 0.0;
    for (size_t i = 0; i < x.size(); ++i) {
        const double d = (double)x[i] - ref[i];
        num += d * d;
        den += (double)ref[i] * ref[i];
    }
    return std::sqrt(num / den);
}

// Independent of cuBLAS: recomputes a fixed sample of entries on the host in
// double precision and compares.
static double spot_error(const std::vector<float> &C, const std::vector<float> &A,
                         const std::vector<float> &B, int n)
{
    std::mt19937 rng(99);
    std::uniform_int_distribution<int> pick(0, n - 1);
    double num = 0.0, den = 0.0;
    for (int s = 0; s < 512; ++s) {
        const int i = pick(rng), j = pick(rng);
        double ref = 0.0;
        for (int k = 0; k < n; ++k) ref += (double)A[(size_t)i * n + k] * B[(size_t)k * n + j];
        const double d = C[(size_t)i * n + j] - ref;
        num += d * d;
        den += ref * ref;
    }
    return std::sqrt(num / den);
}

int main(int argc, char **argv)
{
    std::vector<int> sizes = {1024, 2048, 4096};
    std::string only, out_dir = "results";
    int reps = 10;
    bool profile = false;

    for (int i = 1; i < argc; ++i) {
        const std::string arg = argv[i];
        auto next = [&]() -> std::string {
            if (i + 1 >= argc) {
                std::fprintf(stderr, "missing value for %s\n", arg.c_str());
                std::exit(1);
            }
            return argv[++i];
        };
        if (arg == "--sizes") {
            sizes.clear();
            std::stringstream ss(next());
            for (std::string tok; std::getline(ss, tok, ',');) sizes.push_back(std::atoi(tok.c_str()));
        } else if (arg == "--kernel") {
            only = next();
        } else if (arg == "--reps") {
            reps = std::atoi(next().c_str());
        } else if (arg == "--out") {
            out_dir = next();
        } else if (arg == "--profile") {
            profile = true;
        } else {
            std::fprintf(stderr,
                         "usage: %s [--sizes N1,N2,..] [--kernel NAME] [--reps R] [--out DIR] [--profile]\n"
                         "kernels: cublas naive coalesced tiled regblock vectorized\n",
                         argv[0]);
            return arg == "--help" ? 0 : 1;
        }
    }

    cudaDeviceProp prop;
    CHECK_CUDA(cudaGetDeviceProperties(&prop, 0));
    CHECK_CUBLAS(cublasCreate(&g_cublas));

    // --profile: exactly one launch of one kernel and nothing else on the GPU,
    // so an ncu report of the process contains only that kernel.
    if (profile) {
        if (only.empty() || sizes.size() != 1) {
            std::fprintf(stderr, "--profile needs --kernel NAME and a single --sizes N\n");
            return 1;
        }
    } else {
        std::printf("GPU: %s (%d SMs)\n", prop.name, prop.multiProcessorCount);
    }

    FILE *csv = nullptr;
    if (!profile) {
        std::filesystem::create_directories(out_dir);
        const double bw = measure_bandwidth_gbs();
        const double peak = measure_peak_gflops(prop);
        std::printf("Measured ceilings: %.1f GB/s DRAM bandwidth, %.0f GFLOP/s FMA peak\n\n", bw, peak);

        FILE *dev = std::fopen((out_dir + "/device.csv").c_str(), "w");
        if (!dev) { std::perror("device.csv"); return 1; }
        std::fprintf(dev, "key,value\ngpu,%s\nbandwidth_gbs,%.3f\npeak_gflops,%.3f\n", prop.name, bw, peak);
        std::fclose(dev);

        csv = std::fopen((out_dir + "/bench.csv").c_str(), "w");
        if (!csv) { std::perror("bench.csv"); return 1; }
        std::fprintf(csv, "kernel,n,ms_best,ms_median,gflops,pct_cublas,rel_err,host_err\n");
    }

    bool all_ok = true;
    for (int n : sizes) {
        const size_t count = (size_t)n * n, bytes = count * sizeof(float);
        std::vector<float> hA(count), hB(count), hC(count), hRef;
        std::mt19937 rng(1234);
        std::uniform_real_distribution<float> dist(-1.0f, 1.0f);
        for (auto &v : hA) v = dist(rng);
        for (auto &v : hB) v = dist(rng);

        float *dA, *dB, *dC;
        CHECK_CUDA(cudaMalloc(&dA, bytes));
        CHECK_CUDA(cudaMalloc(&dB, bytes));
        CHECK_CUDA(cudaMalloc(&dC, bytes));
        CHECK_CUDA(cudaMemcpy(dA, hA.data(), bytes, cudaMemcpyHostToDevice));
        CHECK_CUDA(cudaMemcpy(dB, hB.data(), bytes, cudaMemcpyHostToDevice));
        const Problem p = {n, n, n, dA, dB, dC};
        const double gflop = 2.0 * n * n * (double)n / 1e9;

        if (!profile)
            std::printf("N = %d\n  %-11s %10s %10s %10s %9s %10s %9s\n", n, "kernel", "best ms", "median ms",
                        "GFLOP/s", "% cuBLAS", "vs cuBLAS", "vs host");

        double cublas_gflops = 0.0;
        for (const Stage &s : kStages) {
            const bool is_ref = std::string(s.name) == "cublas";
            if (profile ? only != s.name : (!only.empty() && only != s.name && !is_ref)) continue;
            if (n % s.multiple != 0) {
                std::printf("  %-11s skipped (N must be a multiple of %d)\n", s.name, s.multiple);
                continue;
            }

            if (profile) {
                s.run(p);
                CHECK_CUDA(cudaDeviceSynchronize());
                continue;
            }

            s.run(p); // warm-up
            CHECK_CUDA(cudaDeviceSynchronize());
            const Timing t = time_gpu([&] { s.run(p); }, reps);
            CHECK_CUDA(cudaMemcpy(hC.data(), dC, bytes, cudaMemcpyDeviceToHost));

            const double gflops = gflop / (t.best_ms * 1e-3);
            double err = 0.0;
            if (is_ref) {
                hRef = hC;
                cublas_gflops = gflops;
            } else {
                err = rel_error(hC, hRef);
            }
            const double host_err = spot_error(hC, hA, hB, n);
            const bool ok = err < 1e-4 && host_err < 1e-4;
            all_ok = all_ok && ok;
            const double pct = 100.0 * gflops / cublas_gflops;
            std::printf("  %-11s %10.3f %10.3f %10.1f %8.1f%% %10.1e %9.1e%s\n", s.name, t.best_ms,
                        t.median_ms, gflops, pct, err, host_err, ok ? "" : "  WRONG RESULT");
            std::fprintf(csv, "%s,%d,%.4f,%.4f,%.2f,%.2f,%.3e,%.3e\n", s.name, n, t.best_ms, t.median_ms,
                         gflops, pct, err, host_err);
        }
        if (!profile) std::printf("\n");

        CHECK_CUDA(cudaFree(dA));
        CHECK_CUDA(cudaFree(dB));
        CHECK_CUDA(cudaFree(dC));
    }

    if (csv) {
        std::fclose(csv);
        std::printf("Wrote %s/bench.csv and %s/device.csv\n", out_dir.c_str(), out_dir.c_str());
    }
    CHECK_CUBLAS(cublasDestroy(g_cublas));
    return all_ok ? 0 : 1;
}


In [ ]:
%%writefile scripts/profile.sh
#!/usr/bin/env bash
# Profiles each stage with Nsight Compute and exports the counters the roofline
# needs (DRAM bytes moved, floating-point instructions executed).
#
#   scripts/profile.sh [N]        N defaults to 2048
#   FULL=1 scripts/profile.sh     also saves a --set full report per kernel
#                                 (open results/ncu/<kernel>_full.ncu-rep in the ncu UI)
set -euo pipefail

N="${1:-2048}"
BIN="${BIN:-build/sgemm}"
OUT="${OUT:-results}/ncu"
KERNELS="${KERNELS:-naive coalesced tiled regblock vectorized cublas}"

METRICS="dram__bytes_read.sum,dram__bytes_write.sum"
METRICS+=",smsp__sass_thread_inst_executed_op_fadd_pred_on.sum"
METRICS+=",smsp__sass_thread_inst_executed_op_fmul_pred_on.sum"
METRICS+=",smsp__sass_thread_inst_executed_op_ffma_pred_on.sum"
METRICS+=",gpu__time_duration.sum"

mkdir -p "$OUT"
for k in $KERNELS; do
    echo "== $k (N=$N)"
    ncu --metrics "$METRICS" -f -o "$OUT/$k" "$BIN" --profile --kernel "$k" --sizes "$N" > "$OUT/$k.log"
    ncu --import "$OUT/$k.ncu-rep" --csv --page raw --print-units base > "$OUT/$k.csv"
    if [[ "${FULL:-0}" == 1 ]]; then
        ncu --set full -f -o "$OUT/${k}_full" "$BIN" --profile --kernel "$k" --sizes "$N" >> "$OUT/$k.log"
    fi
done
echo "Reports in $OUT. Next: python3 scripts/roofline.py --size $N"


In [ ]:
%%writefile scripts/roofline.py
#!/usr/bin/env python3
"""Roofline plot and memory-bound / compute-bound classification.

Inputs (all under --results, default "results"):
  device.csv      measured ceilings, written by the benchmark
  bench.csv       measured GFLOP/s per kernel, written by the benchmark
  ncu/<k>.csv     Nsight Compute counters per kernel, written by scripts/profile.sh

Arithmetic intensity = FLOPs executed / bytes moved to and from DRAM, both taken
from Nsight Compute. Without ncu data it falls back to the algorithmic minimum
traffic (read A and B once, write C once), which is an upper bound on intensity.
"""
import argparse
import csv
import os
import sys

ORDER = ["naive", "coalesced", "tiled", "regblock", "vectorized", "cublas"]
FLOP_WEIGHTS = {  # an FMA is a multiply and an add
    "smsp__sass_thread_inst_executed_op_fadd_pred_on.sum": 1,
    "smsp__sass_thread_inst_executed_op_fmul_pred_on.sum": 1,
    "smsp__sass_thread_inst_executed_op_ffma_pred_on.sum": 2,
}
BYTE_METRICS = ["dram__bytes_read.sum", "dram__bytes_write.sum"]


def to_float(text):
    try:
        return float(text.replace(",", ""))
    except (AttributeError, ValueError):
        return None


def read_ncu(path):
    """Returns {metric: total over all kernel launches in the report}.

    Handles both ncu CSV layouts: long (Metric Name / Metric Value columns) and
    wide (one column per metric, with a units row under the header).
    """
    totals = {}
    with open(path, newline="") as f:
        rows = list(csv.reader(f))
    # ncu may print warnings before the table; the header is the row with "ID".
    start = next((i for i, r in enumerate(rows) if r and r[0] == "ID"), None)
    if start is None:
        return totals
    header, body = rows[start], rows[start + 1:]
    if "Metric Name" in header:
        name_col, value_col = header.index("Metric Name"), header.index("Metric Value")
        for r in body:
            if len(r) > value_col and to_float(r[value_col]) is not None:
                totals[r[name_col]] = totals.get(r[name_col], 0.0) + to_float(r[value_col])
    else:
        for col, name in enumerate(header):
            for r in body:
                if len(r) > col and to_float(r[col]) is not None:
                    totals[name] = totals.get(name, 0.0) + to_float(r[col])
    return totals


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--results", default="results")
    ap.add_argument("--size", type=int, default=2048, help="matrix size N to plot (must be in bench.csv)")
    ap.add_argument("--out", default=None, help="output image (default: <results>/roofline_N.png)")
    args = ap.parse_args()

    with open(os.path.join(args.results, "device.csv"), newline="") as f:
        device = {r["key"]: r["value"] for r in csv.DictReader(f)}
    bandwidth = float(device["bandwidth_gbs"])  # GB/s
    with open(os.path.join(args.results, "bench.csv"), newline="") as f:
        gflops = {r["kernel"]: float(r["gflops"]) for r in csv.DictReader(f) if int(r["n"]) == args.size}
    if not gflops:
        sys.exit(f"no rows for N={args.size} in bench.csv")
    # The FMA microbenchmark is an estimate; never draw the roof below something we measured.
    peak = max(float(device["peak_gflops"]), max(gflops.values()))
    ridge = peak / bandwidth  # FLOP/byte where the two roofs meet

    points, used_fallback = [], False
    for k in [k for k in ORDER if k in gflops]:
        ncu_path = os.path.join(args.results, "ncu", k + ".csv")
        ncu = read_ncu(ncu_path) if os.path.exists(ncu_path) else {}
        flops = sum(w * ncu.get(m, 0.0) for m, w in FLOP_WEIGHTS.items())
        nbytes = sum(ncu.get(m, 0.0) for m in BYTE_METRICS)
        if flops > 0 and nbytes > 0:
            ai, source = flops / nbytes, "ncu"
        else:
            ai, source, used_fallback = args.size / 6.0, "algorithmic", True  # 2N^3 / (3 * 4N^2)
        roof = min(peak, ai * bandwidth)
        points.append((k, ai, gflops[k], roof, source))

    print(f"{device.get('gpu', 'GPU')}, N={args.size}")
    print(f"ceilings: {bandwidth:.1f} GB/s, {peak:.0f} GFLOP/s, ridge at {ridge:.1f} FLOP/byte\n")
    print(f"{'kernel':<11} {'FLOP/byte':>10} {'GFLOP/s':>9} {'roof':>9} {'% of roof':>10}  regime")
    for k, ai, perf, roof, source in points:
        regime = "memory-bound" if ai < ridge else "compute-bound"
        note = "" if source == "ncu" else "  (no ncu data)"
        print(f"{k:<11} {ai:>10.2f} {perf:>9.1f} {roof:>9.1f} {100 * perf / roof:>9.1f}%  {regime}{note}")
    if used_fallback:
        print("\nwarning: kernels marked '(no ncu data)' use the algorithmic intensity N/6, which only\n"
              "bounds the real one from above. Run scripts/profile.sh for measured values.")

    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
    except ImportError:
        sys.exit("\nmatplotlib not installed; skipping the plot (pip install matplotlib)")

    lo = min(0.05, min(p[1] for p in points) / 2)
    hi = max(ridge * 8, max(p[1] for p in points) * 2)
    fig, ax = plt.subplots(figsize=(8, 5.5))
    ax.plot([lo, ridge, hi], [lo * bandwidth, peak, peak], color="black", lw=2)
    ax.axvline(ridge, color="gray", ls=":", lw=1)
    ax.text(ridge * 0.9, lo * bandwidth * 1.5, "memory-bound", ha="right", color="gray")
    ax.text(ridge * 1.1, lo * bandwidth * 1.5, "compute-bound", ha="left", color="gray")
    ax.text(lo * 1.3, lo * 1.3 * bandwidth * 1.25, f"{bandwidth:.0f} GB/s", rotation=38,
            rotation_mode="anchor", fontsize=9)
    ax.text(hi, peak * 1.08, f"{peak:.0f} GFLOP/s", ha="right", fontsize=9)
    for k, ai, perf, _, source in points:
        ax.scatter(ai, perf, s=60, zorder=3, marker="o" if source == "ncu" else "x")
        ax.annotate(k, (ai, perf), textcoords="offset points", xytext=(7, -3), fontsize=9)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("Arithmetic intensity (FLOP / DRAM byte)")
    ax.set_ylabel("Performance (GFLOP/s)")
    ax.set_title(f"SGEMM roofline, N={args.size}, {device.get('gpu', 'GPU')}")
    ax.grid(True, which="both", alpha=0.25)
    out = args.out or os.path.join(args.results, f"roofline_{args.size}.png")
    fig.tight_layout()
    fig.savefig(out, dpi=160)
    print(f"\nwrote {out}")


if __name__ == "__main__":
    main()


## 3. Build

In [ ]:
!nvcc -O3 -std=c++17 -lineinfo src/main.cu -o build/sgemm -lcublas && echo built

## 4. Benchmark every stage against cuBLAS

Prints GFLOP/s, percent of cuBLAS and the error against cuBLAS, and measures the roofline ceilings (DRAM bandwidth, peak FMA rate). Takes a minute or two; the naive stages are slow at N=4096.

In [ ]:
!./build/sgemm

In [ ]:
import pandas as pd
bench = pd.read_csv('results/bench.csv')
bench.pivot(index='kernel', columns='n', values='gflops').loc[
    ['naive', 'coalesced', 'tiled', 'regblock', 'vectorized', 'cublas']].round(1)

## 5. Profile with Nsight Compute

Collects DRAM bytes moved and floating-point instruction counts per stage. If `ncu` is missing or lacks permission on this runtime, skip this cell: the roofline below still runs, using the algorithmic intensity instead of the measured one.

In [ ]:
!which ncu && bash scripts/profile.sh 2048 || echo 'ncu unavailable or failed; see results/ncu/*.log'

## 6. Roofline: memory-bound vs compute-bound

In [ ]:
!python3 scripts/roofline.py --size 2048
from IPython.display import Image
Image('results/roofline_2048.png', width=760)

## 7. (Optional) Download the results

In [ ]:
!zip -qr results.zip results
from google.colab import files
files.download('results.zip')